# Audio Visualizer — Google Colab CLI Render
Jalankan sel berurutan. Notebook akan clone repository, memasang dependensi, mengunggah aset, merender lewat EGL, lalu mengunduh MP4.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, subprocess

REPO_URL = 'https://github.com/Qodri588/av.git'
BRANCH = 'master'
root = Path('/content/audio-visualizer')
if root.exists():
    shutil.rmtree(root)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(root)], check=True)
project = root
print('Project:', project)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libegl1 libegl-mesa0 libgl1-mesa-dri
!pip -q install -r "{project / 'requirements-colab.txt'}"

In [ ]:
# CPU-only: gunakan EGL Mesa tanpa pemeriksaan NVIDIA/GPU.
import os, sys
render_env = os.environ.copy()
render_env['MGL_BACKEND'] = 'egl'
render_env['LIBGL_ALWAYS_SOFTWARE'] = '1'
probe = subprocess.run([sys.executable, '-c',
    "import moderngl; c=moderngl.create_standalone_context(backend='egl'); print(c.info['GL_VENDOR']); print(c.info['GL_RENDERER']); c.release()"],
    env=render_env, text=True, capture_output=True, check=True)
print(probe.stdout)
print('Mode render: CPU Mesa')

In [ ]:
# Upload audio + template. Background/center image atau video bersifat opsional.
print('Upload audio, template, dan aset override (jika ada):')
assets = files.upload()
upload_dir = Path('/content/render-input')
upload_dir.mkdir(exist_ok=True)
for name, data in assets.items():
    (upload_dir / Path(name).name).write_bytes(data)
print('Tersimpan:', [p.name for p in upload_dir.iterdir()])

In [ ]:
# Ubah nama di bawah bila deteksi otomatis tidak sesuai.
USE_EXAMPLE_TEMPLATE = False  # True = gunakan coba/temp.json
TEMPLATE = ''                 # contoh: 'template.json'
AUDIO = ''                    # contoh: 'lagu.mp3'
BACKGROUND = ''               # opsional: png/jpg/mp4
CENTER = ''                   # opsional: png/jpg
OUTPUT = 'hasil-render.mp4'
RESOLUTION = '720p'           # CPU testing disarankan 720p
FPS = 30                      # 30 atau 60
SUPERSAMPLING = 1             # 1 cepat, 2 kualitas tinggi
TEST_SECONDS = 10             # None untuk render seluruh audio

def first_with(suffixes):
    return next((p for p in upload_dir.iterdir() if p.suffix.lower() in suffixes), None)
template = project / 'coba' / 'temp.json' if USE_EXAMPLE_TEMPLATE else (upload_dir / TEMPLATE if TEMPLATE else first_with({'.json','.yaml','.yml','.toml'}))
audio = upload_dir / AUDIO if AUDIO else first_with({'.wav','.flac','.mp3','.ogg','.aiff','.aif','.m4a'})
if not template or not template.is_file(): raise FileNotFoundError('Template tidak ditemukan')
if not audio or not audio.is_file(): raise FileNotFoundError('Audio tidak ditemukan')
print('Template:', template, '| Audio:', audio)

In [ ]:
import subprocess
output = Path('/content') / OUTPUT
cmd = ['python', str(project/'main.py'), '--cli', str(template), str(audio), '--resolution', RESOLUTION, '--fps', str(FPS), '--supersampling', str(SUPERSAMPLING), '-o', str(output)]
if TEST_SECONDS is not None: cmd += ['--duration', str(TEST_SECONDS)]
if BACKGROUND: cmd += ['--bg', str(upload_dir/BACKGROUND)]
if CENTER: cmd += ['--center', str(upload_dir/CENTER)]
env = render_env.copy()
env['PYTHONPATH'] = str(project)
print('Menjalankan:', ' '.join(cmd))
subprocess.run(cmd, cwd=project, env=env, check=True)
files.download(str(output))